In [1]:
import numpy as np

np.random.seed(42)

X = np.random.uniform(-2, 2, (400,3))

y = (
    np.sin(X[:,0]) +
    0.5*(X[:,1]**2) -
    0.8*X[:,2]
)

y = y.reshape(-1,1)

print("Input shape:",X.shape)
print("Target shape:",y.shape)

Input shape: (400, 3)
Target shape: (400, 1)


In [2]:
def relu(z):
    return np.maximum(0,z)

def relu_deriv(z):
    return (z>0).astype(float)


def sigmoid(z):
    return 1/(1+np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s*(1-s)


def tanh(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z)**2


def leaky_relu(z,alpha=0.01):
    return np.where(z>0,z,alpha*z)

def leaky_relu_deriv(z,alpha=0.01):
    return np.where(z>0,1,alpha)


def softplus(z):
    return np.log(1+np.exp(z))

def softplus_deriv(z):
    return sigmoid(z)

In [3]:
def initialize_params(layers):
    params = {}
    for l in range(1,len(layers)):
        params["W"+str(l)] = np.random.uniform(
            -0.5,0.5,(layers[l],layers[l-1])
        )
        params["b"+str(l)] = np.zeros((layers[l],1))
    return params

In [4]:
def forward(X,params,activation):
    cache = {}
    A = X.T
    cache["A0"] = A
    L = len(params)//2
    for l in range(1,L):
        W = params["W"+str(l)]
        b = params["b"+str(l)]
        Z = W @ A + b
        A = activation(Z)
        cache["Z"+str(l)] = Z
        cache["A"+str(l)] = A

    WL = params["W"+str(L)]
    bL = params["b"+str(L)]
    ZL = WL @ A + bL
    AL = ZL
    cache["Z"+str(L)] = ZL
    cache["A"+str(L)] = AL
    return AL,cache

In [5]:
def mse(y,yhat):
    return np.mean((yhat.T - y)**2)

In [6]:
def backward(X,y,params,cache,activation_deriv):
    grads = {}
    N = X.shape[0]
    L = len(params)//2
    yhat = cache["A"+str(L)]
    dA = 2*(yhat - y.T)
    for l in reversed(range(1,L+1)):
        Z = cache["Z"+str(l)]
        A_prev = cache["A"+str(l-1)]
        if l==L:
            dZ = dA
        else:
            dZ = dA * activation_deriv(Z)
        grads["dW"+str(l)] = (1/N)*(dZ @ A_prev.T)
        grads["db"+str(l)] = (1/N)*np.sum(dZ,axis=1,keepdims=True)
        if l>1:
            W = params["W"+str(l)]
            dA = W.T @ dZ
    return grads

In [7]:
def grad_norm(g):
    return np.sqrt(np.sum(g**2))

In [8]:
def train_model(layers,activation,activation_deriv):
    params = initialize_params(layers)
    lr = 0.01
    epochs = 1000
    loss200 = None
    losses = []
    for epoch in range(epochs):
        yhat,cache = forward(X,params,activation)
        loss = mse(y,yhat)
        losses.append(loss)
        if epoch == 200:
            loss200 = loss
        grads = backward(X,y,params,cache,activation_deriv)
        for l in range(1,len(layers)):
            params["W"+str(l)] -= lr * grads["dW"+str(l)]
            params["b"+str(l)] -= lr * grads["db"+str(l)]

    first_norm = grad_norm(grads["dW1"])
    last_norm = grad_norm(grads["dW"+str(len(layers)-1)])

    if losses[-1] < losses[0]*0.1:
        observation = "Stable"
    elif losses[-1] < losses[0]:
        observation = "Slow"
    else:
        observation = "Unstable"

    return losses[-1], loss200, first_norm, last_norm, observation

In [9]:
models = {
"A":[3,4,1],
"B":[3,6,6,1],
"C":[3,8,8,8,8,1],
"D":[3,8,8,8,8,8,8,8,8,1]
}

In [10]:
results = []

activations = {
"ReLU": (relu, relu_deriv),
"Sigmoid": (sigmoid, sigmoid_deriv)
}

for name,layers in models.items():
    for act in activations:
        f,fd = activations[act]
        final,loss200,g1,gL,obs = train_model(layers,f,fd)
        results.append([name,act,final,loss200,g1,gL,obs])

In [11]:
import pandas as pd

table = pd.DataFrame(
results,
columns=[
"Model",
"Activation",
"Final Loss",
"Loss @200",
"Grad Norm L1",
"Grad Norm Last",
"Observation"
]
)

table

,Model,Activation,Final Loss,Loss @200,Grad Norm L1,Grad Norm Last,Observation
0,A,ReLU,0.111545,0.492686,0.045217,0.037273,Stable
1,A,Sigmoid,0.415810,1.367625,0.037874,0.063916,Slow
2,B,ReLU,0.094230,1.313780,0.055468,0.041026,Stable
3,B,Sigmoid,0.999787,1.699662,0.228174,0.267797,Slow
4,C,ReLU,0.057146,0.505147,0.028671,0.004873,Stable
5,C,Sigmoid,1.740906,1.741983,0.004393,0.007336,Slow
6,D,ReLU,0.077302,1.737661,0.047787,0.009650,Stable
7,D,Sigmoid,1.743853,1.743853,0.000003,0.000001,Slow


In [12]:
## Reflections

#Did deeper always reduce loss faster?
#Observation: deeper models did not always converge faster.

#Did gradients stay similar across layers?
#Observation: gradients in early layers were often smaller.

#Was training stable for all activations?
#Observation: sigmoid showed slower learning in deeper networks.

#Which activation was most stable?
#Observation: ReLU behaved more stable in deeper models.

#Did some models improve slowly even with same learning rate?
#Observation: Yes. Deeper networks with sigmoid improved slowly.